# ✈️ 04 — Prediction Test

This notebook prepares the unseen `Test.csv` data using the **same cleaning and feature-engineering logic as `01_data_cleaning.ipynb`**, then applies the final model saved from `03_models.ipynb`.

### Workflow

1. Load the raw test data
2. Reproduce the training-data cleaning steps
3. Create the same engineered columns
4. Load the fitted model/pipeline
5. Automatically identify the columns expected by the saved model
6. Predict flight delay (`target`) in minutes
7. Export the final prediction file

> The test set does not contain `target`. No target-derived feature is created.


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

import airportsdata
from zoneinfo import ZoneInfo


## 2. Load the Raw Test Data

Use the same raw-data folder structure as the cleaning notebook.


In [ ]:
test_df = pd.read_csv("data/raw/Test.csv")

print("Rows:", test_df.shape[0])
print("Columns:", test_df.shape[1])

test_df.head()


## 3. Create a Working Copy and Clean Identifiers

The training cleaning notebook uses a separate dataframe for processed data. We do the same here.

`FLTID` is stripped so formatting is consistent with the cleaned training data.


In [ ]:
df_test = test_df.copy()

df_test["FLTID"] = df_test["FLTID"].str.strip()


## 4. Convert Date and Time Columns

The parsing formats are identical to those used for the training data.


In [ ]:
df_test["DATOP"] = pd.to_datetime(
    df_test["DATOP"],
    format="%Y-%m-%d"
)

df_test["STD"] = pd.to_datetime(
    df_test["STD"],
    format="%Y-%m-%d %H:%M:%S"
)

df_test["STA"] = pd.to_datetime(
    df_test["STA"],
    format="%Y-%m-%d %H.%M.%S"
)


## 5. Add Airport Time Zones

The departure and arrival airport codes are mapped to IANA time zones using `airportsdata`.

The historical airport `SXF` receives the same manual mapping used in the training-cleaning notebook.


In [ ]:
airport_tz = airportsdata.load("IATA")

df_test["DEP_timezone"] = df_test["DEPSTN"].map(
    lambda code: airport_tz.get(code, {}).get("tz")
)

df_test["ARR_timezone"] = df_test["ARRSTN"].map(
    lambda code: airport_tz.get(code, {}).get("tz")
)

# Same historical-airport fix used for training
df_test.loc[
    df_test["DEPSTN"] == "SXF",
    "DEP_timezone"
] = "Europe/Berlin"

df_test.loc[
    df_test["ARRSTN"] == "SXF",
    "ARR_timezone"
] = "Europe/Berlin"

print(
    "Departure airports without timezone:",
    df_test.loc[
        df_test["DEP_timezone"].isna(),
        "DEPSTN"
    ].unique()
)

print(
    "Arrival airports without timezone:",
    df_test.loc[
        df_test["ARR_timezone"].isna(),
        "ARRSTN"
    ].unique()
)


## 6. Convert Scheduled Times to UTC

UTC conversion is required before calculating duration and interval features.


In [ ]:
def to_utc(dt, tz):
    if pd.isna(dt) or pd.isna(tz):
        return pd.NaT

    return (
        dt.replace(tzinfo=ZoneInfo(tz))
          .astimezone(ZoneInfo("UTC"))
    )


df_test["STD_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_test["STD"],
        df_test["DEP_timezone"]
    )
]

df_test["STA_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_test["STA"],
        df_test["ARR_timezone"]
    )
]


## 7. Scheduled Flight Duration

`duration` is calculated exactly as in the cleaned training dataset.


In [ ]:
df_test["duration"] = (
    df_test["STA_UTC"] - df_test["STD_UTC"]
).dt.total_seconds() / 60


## 8. Calendar Features

The same weekday, weekend, season, hour, month and year variables used in `Train_cleaned.csv` are recreated here.


In [ ]:
df_test["weekday"] = df_test["STD_UTC"].dt.dayofweek

df_test["weekend"] = df_test["weekday"].map({
    0: True,
    1: False,
    2: False,
    3: False,
    4: True,
    5: True,
    6: True
})

df_test["season"] = df_test["STD_UTC"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

df_test["hour"] = df_test["STD_UTC"].dt.hour
df_test["weekday"] = df_test["STD_UTC"].dt.weekday
df_test["month"] = df_test["STD_UTC"].dt.month
df_test["year"] = df_test["STD_UTC"].dt.year

#Create specific time blocks
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

df_test['time_of_day'] = df_test['hour'].apply(get_time_of_day)


## 9. Route

The route is directional and follows the same format used during training:

`DEPSTN_ARRSTN`


In [ ]:
df_test["route"] = (
    df_test["DEPSTN"]
    + "_"
    + df_test["ARRSTN"]
)


## 10. Rotation Sequence

Flights are grouped by operating date and aircraft code, matching the training cleaning notebook.


In [ ]:
df_test = (
    df_test
    .sort_values(["DATOP", "AC"])
    .copy()
)

df_test["rotation_sequence"] = (
    df_test
    .groupby(["DATOP", "AC"])
    .cumcount()
    + 1
)


## 11. Interval Between Consecutive Flights

For each aircraft code, the previous scheduled arrival is shifted by one row and compared with the next scheduled departure.

The first test observation for each aircraft code will naturally have no previous test-set arrival and therefore a missing `interval`.


In [ ]:
df_test = (
    df_test
    .sort_values(["AC", "STD_UTC"])
    .copy()
)

df_test["previous_STA_UTC"] = (
    df_test
    .groupby("AC")["STA_UTC"]
    .shift(1)
)

df_test["interval"] = (
    df_test["STD_UTC"]
    - df_test["previous_STA_UTC"]
).dt.total_seconds() / 60


## 12. Add Airport Size Features

The same external airport-volume data used in `01_data_cleaning.ipynb` is merged onto the test data.

This creates:

- `DEPSIZE`
- `ARRSIZE`


In [ ]:
airport_size = pd.read_csv(
    "data/raw/airport_volume_airport_locations.csv"
)

airport_size = airport_size[
    ["Orig", "TotalSeats"]
]

# Same missing-airport treatment used in training
missing_airports = pd.DataFrame({
    "Orig": ["BYJ", "GAF", "EBM"],
    "TotalSeats": [
        airport_size["TotalSeats"].quantile(0.25),
        airport_size["TotalSeats"].quantile(0.25),
        airport_size["TotalSeats"].quantile(0.25)
    ]
})

airport_size = pd.concat(
    [airport_size, missing_airports],
    ignore_index=True
)

df_test = pd.merge(
    df_test,
    airport_size,
    how="left",
    left_on="DEPSTN",
    right_on="Orig"
)

df_test.rename(
    columns={"TotalSeats": "DEPSIZE"},
    inplace=True
)

df_test.drop(
    columns=["Orig"],
    inplace=True
)

df_test = pd.merge(
    df_test,
    airport_size,
    how="left",
    left_on="ARRSTN",
    right_on="Orig",
    suffixes=("_y", "_x")
)

df_test.rename(
    columns={"TotalSeats": "ARRSIZE"},
    inplace=True
)

df_test.drop(
    columns=["Orig"],
    inplace=True
)


## 13. Verify the Engineered Test Columns

At this stage, the test data should contain the same engineered predictors created by the cleaning notebook.

`target` is intentionally absent because it is what we want to predict.


In [ ]:
engineered_columns = [
    "DEP_timezone",
    "ARR_timezone",
    "STD_UTC",
    "STA_UTC",
    "duration",
    "weekday",
    "weekend",
    "season",
    "hour",
    "month",
    "year",
    "route",
    "rotation_sequence",
    "previous_STA_UTC",
    "interval",
    "DEPSIZE",
    "ARRSIZE"
]

missing_engineered = [
    col
    for col in engineered_columns
    if col not in df_test.columns
]

print("Missing engineered columns:", missing_engineered)

display(
    df_test[engineered_columns].head()
)


## 14. Load the Final Saved Model

The final fitted pipeline should be saved at the end of `03_models.ipynb` with:

In [ ]:
final_model = joblib.load(
    "models/final_hgbrt_model.pkl"
)

final_model


## 15. Match the Exact Features Used During Training

If the saved pipeline exposes `feature_names_in_`, we use it automatically.

This is safer than manually rewriting the model feature list because it guarantees that prediction receives the same input columns and order used during `.fit()`.

If your saved model does not expose `feature_names_in_`, replace the fallback list below with the exact `features` list from `03_models.ipynb`.


In [ ]:
if hasattr(final_model, "feature_names_in_"):
    model_features = list(
        final_model.feature_names_in_
    )
else:
    # Fallback: replace with the exact feature list from 03_models.ipynb
    model_features = [
        "DEPSTN",
        "ARRSTN",
        "interval",
        "month",
        "year",
        "hour",
        "weekday",
        "duration",
        "DEPSIZE",
        "ARRSIZE"
    ]

print("Features expected by model:")
print(model_features)


## 16. Check Training/Test Compatibility

Before predicting, confirm that every feature required by the saved model exists in the prepared test dataframe.

If anything appears in `missing_features`, return to the corresponding cleaning step rather than silently dropping the feature.


In [ ]:
missing_features = [
    col
    for col in model_features
    if col not in df_test.columns
]

print("Missing model features:", missing_features)

if missing_features:
    raise ValueError(
        f"Test data is missing model features: {missing_features}"
    )

X_test = df_test[
    model_features
].copy()

print("X_test shape:", X_test.shape)
X_test.head()


## 17. Predict `target`

The trained model returns the estimated flight delay in minutes.

Negative predictions are clipped to zero because a negative delay is not meaningful for this target definition.


In [ ]:
predictions = final_model.predict(
    X_test
)

predictions = np.maximum(
    predictions,
    0
)

predictions[:10]


## 18. Restore Original Test Order

Feature engineering requires sorting the test records for rotation and interval calculations.

Before creating the submission, the predictions are attached by row and then sorted back to the original `ID` order.


In [ ]:
df_test["target"] = predictions

prediction_output = (
    df_test[
        [
            "ID",
            "DATOP",
            "FLTID",
            "DEPSTN",
            "ARRSTN",
            "target"
        ]
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

prediction_output.head(10)


## 19. Inspect the Prediction Distribution


In [ ]:
prediction_output["target"].describe()


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    prediction_output["target"],
    bins=50
)

plt.xlabel("Predicted Delay (minutes)")
plt.ylabel("Number of Flights")
plt.title("Distribution of Predicted Flight Delays")

plt.tight_layout()
plt.show()


## 20. Inspect the Largest Predicted Delays

Extreme predictions deserve extra inspection because rare, very long delays were the most difficult cases for the training models.


In [ ]:
prediction_output.sort_values(
    "target",
    ascending=False
).head(20)


## 21. Create the Submission File

The final prediction file contains only the identifier and predicted target.


In [ ]:
submission = prediction_output[
    ["ID", "target"]
].copy()

submission.to_csv(
    "flight_delay_predictions.csv",
    index=False
)

submission.head()


## 22. Final Validation

Before using the file, confirm that:

- every test observation has one prediction
- there are no missing targets
- IDs remain unique
- delay predictions are non-negative


In [ ]:
print(
    "Number of test rows:",
    len(test_df)
)

print(
    "Number of predictions:",
    len(submission)
)

print(
    "Missing predictions:",
    submission["target"].isna().sum()
)

print(
    "Duplicated IDs:",
    submission["ID"].duplicated().sum()
)

print(
    "Minimum prediction:",
    submission["target"].min()
)

print(
    "Maximum prediction:",
    submission["target"].max()
)


## Conclusion

The test data has now passed through the same core cleaning and feature-engineering workflow as the training data before being sent to the fitted model.

This is essential for reliable machine-learning inference: **the model must see predictors with the same meaning, format, and construction at training and prediction time**.

The final file `flight_delay_predictions.csv` contains the predicted flight delay in minutes for each test flight.
